In [ ]:
# torch is installed separately as the AMD ROCm build (kernel: Python 3.12 (ROCm)).
# Do not add torch here: pip would replace it with the CPU-only wheel from PyPI.
%pip install -q transformers sentencepiece sacremoses
# uncomment this when ran commented out for ROCm build
# %pip install torch

In [1]:
import requests
import string
import torch
import torch.nn.functional as F
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM
)
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():     # Apple Silicon
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
    
print('torch version: {}'.format(torch.__version__))
print('Device: {}'.format(DEVICE))

c:\Users\2004t\anaconda3\envs\rocm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch version: 2.14.0+rocm10.1.0
Device: cuda


In [2]:
BOOK_URLS = [
    'https://www.gutenberg.org/cache/epub/84/pg84.txt',    # main site
    'https://gutenberg.pglaf.org/cache/epub/84/pg84.txt',  # official Project Gutenberg mirror
]

for url in BOOK_URLS:
    response = requests.get(url, timeout=30)
    if response.ok:
        break
response.raise_for_status()   # stop here if both sources failed

raw_text = response.content.decode('utf-8')
print('Downloaded from: {}'.format(url))
print('Characters downloaded: {:,}'.format(len(raw_text)))

Downloaded from: https://gutenberg.pglaf.org/cache/epub/84/pg84.txt
Characters downloaded: 446,582


In [3]:
sum_name = "sshleifer/distilbart-cnn-12-6"
sum_tok = AutoTokenizer.from_pretrained(sum_name)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(sum_name).to(DEVICE).eval()

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 358/358 [00:00<00:00, 59385.44it/s]


In [4]:
# The model reads at most 1024 tokens at a time and the book is ~92,000 tokens,
# so the book is split into chunks, each chunk is summarized, and the summaries
# are summarized again until everything fits in a single pass.

# Keep only the story (drop the Gutenberg header, table of contents, and licence)
start = raw_text.find('*** START OF')
start = raw_text.find('\n', start)
end = raw_text.find('*** END OF')
book_text = '\n'.join(raw_text[start:end].splitlines()).strip()   # also normalizes \r\n line endings

first_letter = book_text.find('Letter 1')                          # inside the table of contents
book_text = book_text[book_text.find('Letter 1', first_letter + 1):]


def split_into_chunks(text, max_tokens=900):
    """Group paragraphs into chunks that fit in the model's 1024-token window."""
    chunks, current, current_len = [], [], 0
    for para in text.split('\n\n'):
        para = ' '.join(para.split())
        if not para:
            continue
        n = len(sum_tok.encode(para, add_special_tokens=False))
        if current and current_len + n > max_tokens:
            chunks.append(' '.join(current))
            current, current_len = [], 0
        current.append(para)
        current_len += n
    if current:
        chunks.append(' '.join(current))
    return chunks


def summarize(texts, max_length=150, min_length=40, num_beams=4, batch_size=8):
    """Summarize a list of texts, a batch at a time."""
    summaries = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        # max_length here is the model's input limit (1024 tokens); it cannot be raised
        inputs = sum_tok(batch, return_tensors='pt', truncation=True,
                         max_length=1024, padding=True).to(DEVICE)
        with torch.no_grad():
            out = sum_model.generate(
                **inputs,
                num_beams=num_beams,
                max_length=max_length,
                min_length=min_length,
                length_penalty=2.0,
                no_repeat_ngram_size=3,
                early_stopping=True,
            )
        summaries += [s.strip() for s in sum_tok.batch_decode(out, skip_special_tokens=True)]
    return summaries


def summarize_book(text, final_max_length=400, final_min_length=150):
    """Summarize chunk by chunk, then summarize the summaries until one chunk is left."""
    level = 1
    while True:
        chunks = split_into_chunks(text)
        print('Pass {}: {} chunk(s)'.format(level, len(chunks)))
        if len(chunks) == 1:
            return summarize(chunks, max_length=final_max_length, min_length=final_min_length)[0]
        text = '\n\n'.join(summarize(chunks))
        level += 1

In [5]:
import textwrap

print('Original:', len(book_text.split()), 'words')
summary = summarize_book(book_text)
print('Summary: ', len(summary.split()), 'words')
print(textwrap.fill(summary, width=100))

Original: 74975 words
Pass 1: 116 chunk(s)
Pass 2: 12 chunk(s)
Pass 3: 2 chunk(s)
Pass 4: 1 chunk(s)
Summary:  126 words
Genevese was respected by all who knew him for his integrity and indefatigable attention to public
business . He may not see his native land for many years . A sledge drifted towards us in the night
on a large fragment of ice, but only one dog remained alive . A new spirit of life animated the
decaying frame of the stranger . The story follows a journey across the Arctic Ocean by a giant
monster . Frankenstein's remains lay in the cabin where he was murdered . “I am an abortion, to be
spurned at, and kicked, and trampled on," he writes . ‘I am . an abortion" he writes in his book "
Frankenstein's Remains of Frankenstein" The film is based on the novel
